# Notebook 05: RAG (Retrieval-Augmented Generation) Layer

This notebook demonstrates a lightweight RAG pipeline:

1. **Retrieve** — find relevant papers using embeddings (using our recommender)
2. **Augment** — format retrieved papers as LLM context
3. **Generate** — produce natural language summaries and insights

---

### LLM Backend Options

This notebook supports multiple backends:
- **OpenAI** — requires `OPENAI_API_KEY` environment variable
- **Google Gemini** — requires `GOOGLE_API_KEY` environment variable

If no LLM is available, the retrieval step still works — you just won't get generated summaries.

We will be using a sampled subset of 100k as anything more than that would need to compute for 30 minutes and above.

All the LLM responses will be pasted to a markdown cell at the end of this notebook as a part of the summary.

In [1]:
import sys, os
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path(os.getcwd()).parent if 'notebooks' in os.getcwd() else Path(os.getcwd())
sys.path.insert(0, str(PROJECT_ROOT))

from src.parser import load_from_parquet
from src.embeddings import load_or_compute_embeddings
from src.recommender import ContentRecommender

# Load data
df = load_from_parquet(PROJECT_ROOT / "data" / "processed" / "dblp_subset.parquet")
df_rec = df.dropna(subset=['title']).copy()
df_rec = df_rec[df_rec['title'].str.len() > 10].reset_index(drop=True)

RECOMMENDER_SIZE = min(100_000, len(df_rec))
df_rec = df_rec.sample(n=RECOMMENDER_SIZE, random_state=42).reset_index(drop=True)
titles = df_rec['title'].tolist()

# Load embeddings
CACHE_DIR = PROJECT_ROOT / "data" / "processed"
embeddings = load_or_compute_embeddings(
    titles,
    cache_path=CACHE_DIR / "embeddings_recommender.npy",
    method="transformer",
    model_name="all-MiniLM-L6-v2",
)

# Initialize recommender
recommender = ContentRecommender(df=df_rec, embeddings=embeddings)
print(f"RAG system ready with {len(df_rec):,} papers")

Loading cached embeddings from d:\Rekrutacja_nokia\data\processed\embeddings_recommender.npy
Loaded embeddings: shape (100000, 384)
Recommender initialized with 100,000 papers, embedding dim=384
RAG system ready with 100,000 papers


## 6.1 Retrieval Step

Let's see what papers are retrieved for sample queries.

In [2]:
from src.rag import retrieve_context, build_prompt
from dotenv import load_dotenv
load_dotenv()

query = "What are recent trends in graph neural networks?"

results, context = retrieve_context(query, recommender, top_k=5)

print(f"Query: \"{query}\"\n")
print("Retrieved Papers:")
print(context)

print("\n" + "="*70)
print("\nFormatted as DataFrame:")
results[['title', 'year', 'venue', 'similarity_score']]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Query: "What are recent trends in graph neural networks?"

Retrieved Papers:
[1] Title: On the Bottleneck of Graph Neural Networks and its Practical Implications.
    Authors: Uri Alon 0002, Eran Yahav
    Year: 2020 | Venue: CoRR
    Relevance Score: 0.6679

[2] Title: Graph Neural Networks in Computer Vision - Architectures, Datasets and Common Approaches.
    Authors: Maciej Krzywda, Szymon Lukasik, Amir H. Gandomi
    Year: 2022 | Venue: IJCNN
    Relevance Score: 0.6290

[3] Title: Graph Neural Networks in Computer Vision - Architectures, Datasets and Common Approaches.
    Authors: Maciej Krzywda, Szymon Lukasik, Amir H. Gandomi
    Year: 2022 | Venue: CoRR
    Relevance Score: 0.6290

[4] Title: DropGNN: Random Dropouts Increase the Expressiveness of Graph Neural Networks.
    Authors: Pál András Papp, Karolis Martinkus, Lukas Faber, Roger Wattenhofer
    Year: 2021 | Venue: NeurIPS
    Relevance Score: 0.6106

[5] Title: Scalable Spatiotemporal Graph Neural Networks.
    Author

,title,year,venue,similarity_score
0,On the Bottleneck of Graph Neural Networks and...,2020,CoRR,0.667896
1,Graph Neural Networks in Computer Vision - Arc...,2022,IJCNN,0.629018
2,Graph Neural Networks in Computer Vision - Arc...,2022,CoRR,0.629018
3,DropGNN: Random Dropouts Increase the Expressi...,2021,NeurIPS,0.610567
4,Scalable Spatiotemporal Graph Neural Networks.,2022,CoRR,0.608955


## 6.2 Prompt Construction

The prompt is structured to give the LLM clear context and instructions.

In [3]:
# Build prompts for different tasks
prompt_summarize = build_prompt(query, context, task='summarize')
prompt_trends = build_prompt(query, context, task='trends')

print("=== SUMMARIZE PROMPT ===")
print(prompt_summarize[:500])
print("...\n")

print("=== TRENDS PROMPT ===")
print(prompt_trends[:500])
print("...")

=== SUMMARIZE PROMPT ===
You are a research assistant analyzing computer science publications from DBLP.

Based on the following retrieved papers relevant to the query "What are recent trends in graph neural networks?":

[1] Title: On the Bottleneck of Graph Neural Networks and its Practical Implications.
    Authors: Uri Alon 0002, Eran Yahav
    Year: 2020 | Venue: CoRR
    Relevance Score: 0.6679

[2] Title: Graph Neural Networks in Computer Vision - Architectures, Datasets and Common Approaches.
    Authors: Maciej 
...

=== TRENDS PROMPT ===
You are a research assistant analyzing computer science publications from DBLP.

Based on the following retrieved papers relevant to the query "What are recent trends in graph neural networks?":

[1] Title: On the Bottleneck of Graph Neural Networks and its Practical Implications.
    Authors: Uri Alon 0002, Eran Yahav
    Year: 2020 | Venue: CoRR
    Relevance Score: 0.6679

[2] Title: Graph Neural Networks in Computer Vision - Architectures,

## 6.3 Generate Response with LLM

Choose your LLM backend below. If none is available, the retrieval context
above already provides valuable results.

In [ ]:
from src.rag import rag_query

# Choose one of:  'openai', 'gemini'
LLM_BACKEND = 'gemini'  # Change this based on your setup

# Backend-specific kwargs
llm_kwargs = {}
if LLM_BACKEND == 'openai':
    llm_kwargs = {'model': 'gpt-3.5-turbo', 'api_key': os.environ.get('OPENAI_API_KEY')}
elif LLM_BACKEND == 'gemini':
    llm_kwargs = {'model': 'gemini-2.5-flash', 'api_key': os.environ.get('GOOGLE_API_KEY')}

print(f"Using LLM backend: {LLM_BACKEND}")

Using LLM backend: gemini


In [4]:
# Run full RAG pipeline
queries = [
    "What are the dominant research trends in machine learning and artificial intelligence based on recent publications",
    "What are the key research directions and challenges in computer vision based on recent publications?",
    "What are the main research areas in computer science today, and how do emerging topics differ from more established ones?",
]


print("\n" + "="*80)
print(f"QUERY: {queries[0]}")
print("="*80)

result = rag_query(
    queries[0], 
    recommender,
    llm_backend=LLM_BACKEND,
    task='summarize',
    top_k=15,
    **llm_kwargs
)

print(f"\nRetrieved {len(result['retrieved_papers'])} papers")
print(f"\nLLM Response:")
print(result['response'])


QUERY: What are the dominant research trends in machine learning and artificial intelligence based on recent publications


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Retrieved 15 papers

LLM Response:
Gemini API error: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-3-flash is not found for API version v1beta, or is not supported for generateContent. Call ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}


In [7]:
print("\n" + "="*80)
print(f"QUERY: {queries[1]}")
print("="*80)

result = rag_query(
    queries[1], 
    recommender,
    llm_backend=LLM_BACKEND,
    task='summarize',
    top_k=5,
    **llm_kwargs
)

print(f"\nRetrieved {len(result['retrieved_papers'])} papers")
print(f"\nLLM Response:")
print(result['response'])


QUERY: How has natural language processing evolved with transformers?


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Retrieved 5 papers

LLM Response:
The provided papers illustrate key facets of how natural language processing (NLP) has evolved with the advent and widespread adoption of transformers. The research spans the application, optimization, and foundational understanding of these powerful models, showcasing a clear progression in the field.

**Key Findings and Contributions:**

1.  **Enabling Advanced NLP Applications:**
    *   Paper [1] demonstrates the capability of transformer networks in **Natural Language Generation (NLG)**, specifically in an open-domain setting. This highlights how transformers have advanced the ability to generate coherent and contextually relevant text, a significant step in NLP.

2.  **Broadening Linguistic Scope and Accessibility:**
    *   Paper [2] focuses on the development of **Transformer-Based Language Models for Bulgarian**. This shows how transformers are extending state-of-the-art NLP capabilities beyond high-resource languages, making advanced models 

In [8]:
print("\n" + "="*80)
print(f"QUERY: {queries[2]}")
print("="*80)

result = rag_query(
    queries[2], 
    recommender,
    llm_backend=LLM_BACKEND,
    task='summarize',
    top_k=5,
    **llm_kwargs
)

print(f"\nRetrieved {len(result['retrieved_papers'])} papers")
print(f"\nLLM Response:")
print(result['response'])


QUERY: What privacy-preserving techniques are used in machine learning?


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Retrieved 5 papers

LLM Response:
The provided papers offer insights into various privacy-preserving techniques used in machine learning, covering general risk assessment, specific model implementations, and different data distribution scenarios.

Here are the key findings and contributions, highlighting common themes:

**1. Understanding and Mitigating Privacy Risk:**
*   **General Risk and Mitigation:** Paper [1] provides a foundational understanding of privacy risks inherent in machine learning systems and explores strategies for their mitigation. This paper likely covers a broad spectrum of techniques and risk factors.
*   **Severity of Attacks:** Paper [2] delves into the critical question of "How Worrying Are Privacy Attacks Against Machine Learning?", thereby emphasizing the urgent need for robust privacy-preserving techniques. While not proposing a technique itself, it highlights the significance and impact of privacy breaches, contextualizing the work of other papers.

**2. P


## QUERY: What are recent trends in graph neural networks?
